# Metodología de recuperación de CatRAG-Reif, paso a paso

Este cuaderno recorre **la fase de consulta** del sistema evaluado en 2WikiMultihopQA sobre el índice real
del banco de pruebas (grafo canónico de 81 058 nodos), con las mismas preguntas que usa el documento
`docs/metodologia_2wiki.tex`. Cada etapa imprime lo que produce y lo que entrega a la siguiente:

| etapa | sección del documento | qué produce |
|---|---|---|
| 0. El índice en dos vistazos | §2.3 | entrada canónica, aserción reificada, grafo por capas |
| 1. Enrutador léxico | §2.4.1 | tipo y estrategia |
| 2. Enlazador de menciones | §2.4.2 | $L(q)$ (rama simbólica) |
| 3. Vectores de consulta y afinidad | §2.4.3 | $\{v_j\}$, $\sigma_a$ (rama semántica) |
| 4. Candidatos $\mathcal K(q)$ y filtro selector | §2.4.4 | $S(q)$, $\tilde\sigma$ |
| 5. Siembra (cuatro fuentes) | §2.4.5 | vector $\mathbf v$ |
| 6. Paseo con transiciones moduladas | §2.4.6 | orden de pasajes |
| 7. Comparación con HippoRAG 2 en las mismas preguntas | §5.3 | |

**Cómo ejecutarlo.** Desde la raíz del repositorio, con el entorno `venv` y el fichero `.env` (clave de OpenAI):
la carga del grafo tarda 1–2 minutos; cada pregunta trazada hace una o dos llamadas a `gpt-4o-mini`
(filtro o descomposición) y una incrustación: céntimos. Las tres preguntas ya vienen ejecutadas con sus salidas.
Las funciones de traza reproducen exactamente lo que hace `Wiki2CatRAG.buscar()` (`src/asistente_vih/retrieval/wiki2_catrag.py`)
y se comprueban contra él.

In [2]:
import os, sys, json, re
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists():          # permite ejecutar desde notebooks/ o desde la raíz
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 80)

from asistente_vih.retrieval.wiki2_catrag import Wiki2CatRAG, spans_entidad, _plegar_alias, clasificar_tipo
from asistente_vih.eval.wiki2 import cargar_benchmark, titulos_oro

preguntas, corpus = cargar_benchmark()
print(f"banco: {len(preguntas)} preguntas · corpus: {len(corpus)} pasajes")

banco: 1000 preguntas · corpus: 6119 pasajes


In [5]:
corpus[1]

{'title': 'Theodred II (Bishop of Elmham)',
 'text': "Theodred II was a medieval Bishop of Elmham. The date of Theodred's consecration unknown, but the date of his death was sometime between 995 and 997."}

In [4]:
preguntas[3]

{'_id': '462bb642099211ebbdb0ac1f6bf848b6',
 'type': 'comparison',
 'question': 'Are Marufabad and Nasamkhrali both located in the same country?',
 'context': [['Limestone Coast',
   ['The Limestone Coast is a name used since the early twenty- first century for a South Australian government region located in the south east of South Australia which immediately adjoins the continental coastline and the Victorian border.',
    'The name is also used for a tourist region and a wine zone both located in the same part of South Australia.']],
  ['Marufabad',
   ['Marufabad( also Romanized as Ma‘rūfābād) is a village in Kabutarsorkh Rural District, in the Central District of Chadegan County, Isfahan Province, Iran.',
    'At the 2006 census, its population was 545, in 134 families.']],
  ['Telephone numbers in Ascension Island',
   ['Country Code:+ 247< br> International Call Prefix: 00 Ascension Island does not share the same country code( +290) with the rest of St Helena.']],
  ['Jawty',
   

## Preparación: el recuperador tal como se evaluó

`split="benchmark"` (corpus de 6 119 pasajes), `variante="router"` (enrutador léxico: comparación → descomposición; resto → filtro selector),
`modo="paridad"` (diales publicados de HippoRAG 2: $\lambda=0{,}5$, peso denso $0{,}05$, todos los pasajes sembrados) y
`canonico=True` (grafo sobre la memoria canónica de entidades, enlazador y pasaje propio). Es la **v3** del documento (R@5 96,0 / FC@5 90,6).

In [6]:
r = Wiki2CatRAG(split="benchmark", variante="router", modo="paridad", canonico=True)
n_ent = sum(1 for n in r.nodos if n.startswith("can:")); n_chunk = sum(1 for n in r.nodos if n.startswith("chunk:"))
print(f"grafo de búsqueda: {len(r.nodos)} nodos = {n_ent} entidades canónicas + {len(r.aser_ids)} aserciones + {n_chunk} pasajes; {r.A.nnz} aristas dirigidas")
print(f"diales: peso denso w_pas={r.peso_pasaje} · amortiguación λ={r.damping} · ancla débil ε={r.eps} · ancla del enlazador w_enl={r.peso_enlace}")
print(f"memoria canónica: {len(r.entradas)} entradas, {len(r.alias_idx)} alias plegados; fichas incrustadas: {r.ent_emb.shape}")

grafo de búsqueda: 81058 nodos = 26934 entidades canónicas + 47999 aserciones + 6119 pasajes; 165240 aristas dirigidas
diales: peso denso w_pas=0.05 · amortiguación λ=0.5 · ancla débil ε=0.05 · ancla del enlazador w_enl=0.5
memoria canónica: 26959 entradas, 30666 alias plegados; fichas incrustadas: (26960, 1536)


## 0. El índice en dos vistazos (§2.3)

Una **entrada canónica** agrupa todas las formas con que el corpus nombra a una entidad, y guarda su *pasaje propio* (el artículo cuyo sujeto es).
Una **aserción reificada** es un hecho promovido a nodo: frase autocontenida, procedencia y un rol por participante.

In [7]:
def entrada(eid):
    e = r.entradas[eid]
    return {k: e[k] for k in ("id", "nombre", "alias", "pasajes", "pasaje_propio", "n_menciones")} | {"descripcion": e["descripcion"][:160] + "…"}

for eid in ["can:h. p. lovecraft", "can:robert a. stemmle"]:
    print(json.dumps(entrada(eid), ensure_ascii=False, indent=1)); print()

{
 "id": "can:h. p. lovecraft",
 "nombre": "H. P. Lovecraft",
 "alias": [
  "H. P. Lovecraft",
  "Howard Phillips Lovecraft"
 ],
 "pasajes": [
  "H. P. Lovecraft",
  "Sonia Greene"
 ],
 "pasaje_propio": "H. P. Lovecraft",
 "n_menciones": 3,
 "descripcion": "Howard Phillips Lovecraft was born on August 20, 1890 Howard Phillips Lovecraft died on March 15, 1937 Howard Phillips Lovecraft was born in Providence, Rhode I…"
}

{
 "id": "can:robert a. stemmle",
 "nombre": "Robert A. Stemmle",
 "alias": [
  "Robert A. Stemmle",
  "Robert Adolf Stemmle"
 ],
 "pasajes": [
  "Robert A. Stemmle",
  "You Can No Longer Remain Silent"
 ],
 "pasaje_propio": "Robert A. Stemmle",
 "n_menciones": 3,
 "descripcion": "Robert Adolf Stemmle was born on 10 June 1903 in Magdeburg, Germany Robert Adolf Stemmle died on 24 February 1974 in Baden-Baden, Germany Robert Adolf Stemmle w…"
}



In [4]:
# una aserción real y sus aristas en el grafo canónico
a1 = next(a for a in r.aser_ids if r.g.nodes[a].get("descripcion", "").startswith("Atomised was directed"))
print("nodo:", a1); print("atributos:", {k: v for k, v in r.g.nodes[a1].items() if k in ("tipo", "descripcion", "relacion", "tipo_relacion")})
for _, v, d in r.g.out_edges(a1, data=True):
    print(f"   {a1} --{d.get('tipo_relacion')}--> {v}")

nodo: Atomised (film)::a1
atributos: {'tipo': 'Asercion', 'descripcion': 'Atomised was directed by Oskar Roehler.'}
   Atomised (film)::a1 --EN_CHUNK--> chunk:Atomised (film)
   Atomised (film)::a1 --HAS_INTERVENTION--> can:atomised
   Atomised (film)::a1 --HAS_OUTCOME--> can:oskar roehler


In [5]:
# el grafo de búsqueda es un grafo por capas: entidad → aserción → pasaje (y entidad → pasaje)
A = r.A.tocsr(); indeg = np.asarray(A.sum(0)).ravel(); outdeg = np.asarray(A.sum(1)).ravel()
es_ent = np.array([n.startswith("can:") for n in r.nodos]); es_chunk = np.array([n.startswith("chunk:") for n in r.nodos])
es_aser = ~es_ent & ~es_chunk
print(f"entidades: grado de entrada máximo {indeg[es_ent].max():.0f} (nunca reciben masa del paseo: su masa es su siembra)")
print(f"aserciones: grado de salida mínimo {outdeg[es_aser].min():.0f} / máximo {outdeg[es_aser].max():.0f} (cada una desagua en su único pasaje)")
print(f"pasajes: grado de salida máximo {outdeg[es_chunk].max():.0f} (sumideros)")

entidades: grado de entrada máximo 0 (nunca reciben masa del paseo: su masa es su siembra)
aserciones: grado de salida mínimo 1 / máximo 13 (cada una desagua en su único pasaje)
pasajes: grado de salida máximo 0 (sumideros)


## Funciones de traza

Cada función reproduce una etapa de `Wiki2CatRAG.buscar()` e imprime sus productos. La siembra se recompone fuente a fuente
—(a) hechos, (b) presa densa, (c) anclas débiles, (d) enlazador y pasaje propio— y se comprueba contra `_semillas()` del sistema.

In [8]:
_INV = ("HAS_INTERVENTION", "HAS_OUTCOME")           # roles de aserción (invertidos en el grafo de búsqueda)
def desc(j): return r.g.nodes[r.aser_ids[j]].get("descripcion", "")
def marca(t, oro): return "ORO" if t in oro else ""
def buscar_pregunta(inicio): return next(q for q in preguntas if q["question"].startswith(inicio))

def etapa_enrutador(Q):
    tipo, est = clasificar_tipo(Q), r._variante_efectiva(Q)
    print(f"tipo léxico: {tipo}  →  estrategia: {est}  ({'filtro selector, 1 llamada' if est == 'juez' else 'descomposición en subconsultas, 1 llamada; sin filtro'})")
    return est

def etapa_enlazador(Q):
    for t in spans_entidad(Q):
        pl = _plegar_alias(t); hit = r.alias_idx.get(pl) or r.alias_idx.get(_plegar_alias(t.split("(")[0]))
        print(f"  tramo {t!r} → plegado {pl!r} → alias exacto: {hit}")
    L = r._enlazar_menciones(Q); r._enlaces_actuales = L
    print("  L(q) =", L); return L

def etapa_afinidad(Q, est, n=8):
    qv = r._qvecs(Q, est)                                   # v_0 = e(q) [+ subconsultas]
    S = np.stack([r.aser_emb @ v for v in qv]); sig = S.max(0); quien = S.argmax(0)
    print(f"  J = {len(qv) - 1} vectores adicionales · σ_a = max_j <v_j, e(τ_a)> sobre {len(sig)} hechos: mín {sig.min():.3f}, máx {sig.max():.3f}")
    filas = [(round(float(sig[j]), 3), f"v_{quien[j]}", desc(j)[:75], [str(e) for e in r.aser_ents[j]]) for j in np.argsort(-sig)[:n]]
    print(pd.DataFrame(filas, columns=["σ", "vector", "hecho τ_a", "entidades"]).to_string(index=False)); return qv, sig

def etapa_candidatos(L, sig, puentes=()):
    orden = np.argsort(-sig); rango = {int(j): k + 1 for k, j in enumerate(orden)}; top40 = [int(j) for j in orden[:40]]; nuevos = []
    for eid in L:
        asers = [r.aser_idx[u] for u, _, d in r.g.in_edges(eid, data=True) if d.get("tipo_relacion") in _INV and u in r.aser_idx]
        asers.sort(key=lambda j: -sig[j]); f15 = asers[:15]; nn = [j for j in f15 if j not in top40]
        print(f"  frontera de {eid}: {len(asers)} hechos incidentes → {len(f15)} de mayor σ, de los que {len(nn)} no estaban en el Top-40 (rangos por coseno: {sorted(rango[j] for j in f15)})")
        nuevos += [j for j in nn if j not in nuevos]
    K = top40 + nuevos[:20]
    print(f"  K(q) = Top-40 ∪ frontera = 40 + {min(len(nuevos), 20)} = {len(K)} candidatos numerados")
    for txt in puentes:
        for i in [i for i in range(len(r.aser_ids)) if desc(i).startswith(txt)]:
            print(f"  hecho puente «{txt[:55]}»: σ = {sig[i]:.3f}, rango {rango[i]}, en Top-40: {i in top40}, en K(q): {i in K}")
    return K

def etapa_filtro(Q, qv, est):
    sims, apr = r._sims_aserciones(Q, qv, est)             # en 'juez' llama al filtro (gpt-4o-mini) con K(q)
    if apr is None: print("  la estrategia de descomposición no ejecuta el filtro: S(q) = ∅ y σ̃ = σ (compuerta blanda en la siembra)")
    else:
        print(f"  S(q): {len(apr)} hechos aprobados; para ellos σ̃ = 1 (μ = 1 en el paseo):")
        for j in apr: print(f"    · {desc(j)[:80]}   ents = {[str(e) for e in r.aser_ents[j]]}")
    return sims, apr

def etapa_siembra(Q, qv, sims, apr, oro, top=10):
    src = {}
    def add(n, k, w): src.setdefault(n, [0.0] * 4); src[n][k] += w
    if apr is not None:                                     # (a) compuerta dura: entidades de los hechos aprobados, peso = media(σ̃)
        pesos, cuenta = {}, {}
        for j in apr:
            for e in r.aser_ents[j]: pesos[e] = pesos.get(e, 0) + float(np.clip(sims[j], 0, 1)); cuenta[e] = cuenta.get(e, 0) + 1
        for e in pesos: add(e, 0, pesos[e] / cuenta[e])
    else:                                                   # (a) compuerta blanda: 15 hechos de mayor σ, media(σ normalizado / frecuencia), 5 mejores
        norm = (sims - sims.min()) / (sims.max() - sims.min() + 1e-9); pesos, cuenta = {}, {}
        for j in np.argsort(-sims)[:15]:
            for e in r.aser_ents[j]:
                f = r.freq_ent.get(e, 1) or 1; pesos[e] = pesos.get(e, 0) + float(norm[j]) / f; cuenta[e] = cuenta.get(e, 0) + 1
        for e in pesos: pesos[e] /= cuenta[e]
        for e, w in sorted(pesos.items(), key=lambda x: -x[1])[:5]: add(e, 0, w)
    s = r.chunk_emb @ qv[0]; mm = (s - s.min()) / (s.max() - s.min() + 1e-9)   # (b) presa densa: todos los pasajes, 0,05 × coseno normalizado con v_0
    for t, m in enumerate(mm):
        n = f"chunk:{r.titulos[t]}"
        if n in r.idx: add(n, 1, r.peso_pasaje * float(m))
    for v in qv:                                            # (c) anclas débiles: 2 fichas por vector, +ε
        se = r.ent_emb @ v
        for j in np.argsort(-se)[:2]:
            n = str(r.ent_ids[int(j)])
            if n in r.idx: add(n, 2, r.eps)
    for eid in r._enlaces_actuales:                         # (d) ancla del enlazador: max(v, w_enl) …
        tot = sum(src.get(eid, [0] * 4))
        if tot < r.peso_enlace: add(eid, 3, r.peso_enlace - tot)
    for eid, vals in list(src.items()):                     # … y pasaje propio: v[π(ε)] += w_enl · v(ε)
        if eid.startswith("chunk:"): continue
        ent = r.entradas.get(eid)
        if ent and ent.get("pasaje_propio"):
            n = f"chunk:{ent['pasaje_propio']}"
            if n in r.idx: add(n, 3, r.peso_enlace * sum(vals))
    tot = {n: sum(v) for n, v in src.items()}; Z = sum(tot.values()); pf = [sum(v[k] for v in src.values()) for k in range(4)]
    print(f"  masa sin normalizar ‖v‖₁ = {Z:.2f}: (a) hechos {pf[0]:.2f} · (b) presa densa {pf[1]:.2f} ({pf[1] / Z * 100:.0f} % de la masa, en {sum(1 for v in src.values() if v[1] > 0)} pasajes) · (c) anclas {pf[2]:.2f} · (d) enlace + pasaje propio {pf[3]:.2f}")
    filas = [(str(n), *[round(x, 3) for x in v], round(tot[n], 3), round(tot[n] / Z, 4), marca(n[6:], oro) if n.startswith("chunk:") else "")
             for n, v in sorted(src.items(), key=lambda x: -tot[x[0]])[:top]]
    print(pd.DataFrame(filas, columns=["nodo", "(a)", "(b)", "(c)", "(d)", "total", "v norm.", ""]).to_string(index=False))
    pers = r._semillas(Q, qv, sims, apr)
    print(f"  comprobación contra _semillas() del sistema: diferencia máxima {max(abs(pers.get(n, 0) - tot.get(n, 0)) for n in set(pers) | set(tot)):.1e}")
    return pers

def filas_transicion(nodo, sims, top=6):
    mult = np.ones(len(r.nodos)); mult[r.aser_pos] = 0.25 + 0.75 * np.clip(sims, 0, 1)   # μ_a = 1/4 + 3/4·σ̃_a
    cols = A.getrow(r.idx[nodo]).indices; mu = mult[cols]; mod = mu / mu.sum()
    print(f"  {nodo}: {len(cols)} salidas · transición FIJA (sin modular) = {1 / len(cols):.3f} para todas · MODULADA:")
    for k in np.argsort(-mod)[:top]:
        v = r.nodos[cols[k]]; d = r.g.nodes[v]
        print(f"    μ = {mu[k]:.3f}   P = {mod[k]:.3f}   " + (("HECHO   " + d.get("descripcion", "")[:60]) if d.get("tipo") == "Asercion" else ("PASAJE  " + v[6:])))

def etapa_paseo(pers, sims, oro, k=10, fijo=False):
    p = r._ppr(pers, np.full(len(sims), 1.0) if fijo else sims)   # fijo=True: μ ≡ 1 (matriz de transición sin modular)
    orden = [i for i in np.argsort(-p) if r.nodos[i].startswith("chunk:")][:k]; top = [r.nodos[i][6:] for i in orden]
    print(pd.DataFrame([(j + 1, t, round(float(p[i]), 4), marca(t, oro)) for j, (i, t) in enumerate(zip(orden, top))],
                       columns=["puesto", "pasaje", "p*", ""]).to_string(index=False)); return top

def metricas(top, oro, ks=(2, 5, 10)):
    print("  " + "   ".join(f"R@{k} = {len(oro & set(top[:k])) / len(oro):.2f}, FC@{k} = {int(len(oro & set(top[:k])) == len(oro))}" for k in ks))

print("funciones de traza definidas")

funciones de traza definidas


## Pregunta conductora del documento (comparación-puente, cuatro pasajes oro)

*«Which film has the director who is older, God'S Gift To Women or Aldri Annet Enn Bråk?»* — el tipo más difícil del banco:
los sistemas publicados no superan el 12,3 % de cadena completa en él.

In [7]:
q = buscar_pregunta("Which film has the director who is older, God'S Gift To Women"); Q = q["question"]; oro = titulos_oro(q)
print(Q); print("pasajes oro:", sorted(oro))

Which film has the director who is older, God'S Gift To Women or Aldri Annet Enn Bråk?
pasajes oro: ['Aldri annet enn bråk', 'Edith Carlmar', "God's Gift to Women", 'Michael Curtiz']


### 1. Enrutador léxico (§2.4.1)

In [8]:
est = etapa_enrutador(Q)

tipo léxico: bridge_comparison  →  estrategia: juez  (filtro selector, 1 llamada)


### 2. Enlazador de menciones (§2.4.2) — rama simbólica

In [9]:
L = etapa_enlazador(Q)

  tramo "God'S Gift To Women" → plegado 'god s gift to women' → alias exacto: can:god's gift to women
  tramo 'Aldri Annet Enn Bråk' → plegado 'aldri annet enn br k' → alias exacto: can:aldri annet enn bråk
  L(q) = ["can:god's gift to women", 'can:aldri annet enn bråk']


### 3. Vectores de consulta y afinidad (§2.4.3) — rama semántica

Con la estrategia de filtro, $J=0$: un solo vector. Obsérvese el sesgo del coseno hacia la rama léxicamente dominante.

In [10]:
qv, sig = etapa_afinidad(Q, est)

  J = 0 vectores adicionales · σ_a = max_j <v_j, e(τ_a)> sobre 47999 hechos: mín -0.088, máx 0.554
    σ vector                                                       hecho τ_a                                         entidades
0.554    v_0             God's Gift to Women was directed by Michael Curtiz.     [can:god's gift to women, can:michael curtiz]
0.517    v_0 God's Gift to Women was originally completed as a musical film.                         [can:god's gift to women]
0.517    v_0                       God's Gift to Women was released in 1931.                         [can:god's gift to women]
0.507    v_0                        God's Gift to Women stars Joan Blondell.      [can:god's gift to women, can:joan blondell]
0.480    v_0     Edith Carlmar is known for the film 'Aldri annet enn bråk'.     [can:edith carlmar, can:aldri annet enn bråk]
0.477    v_0    God's Gift to Women is based on the play The Devil Was Sick. [can:god's gift to women, can:the devil was sick]
0.470    v_0

### 4. Candidatos $\mathcal K(q)$ = Top-40 ∪ frontera, y filtro selector (§2.4.4)

Las dos ramas convergen aquí: el Top-40 viene de $\sigma$, la frontera de $L(q)$ (hechos incidentes en cada entidad enlazada).

In [11]:
K = etapa_candidatos(L, sig, puentes=("God's Gift to Women was directed by Michael Curtiz", "Edith Carlmar is known for the film 'Aldri"))

  frontera de can:god's gift to women: 9 hechos incidentes → 9 de mayor σ, de los que 2 no estaban en el Top-40 (rangos por coseno: [1, 2, 3, 4, 6, 8, 17, 57, 6742])
  frontera de can:aldri annet enn bråk: 7 hechos incidentes → 7 de mayor σ, de los que 5 no estaban en el Top-40 (rangos por coseno: [5, 12, 114, 165, 239, 584, 1416])
  K(q) = Top-40 ∪ frontera = 40 + 7 = 47 candidatos numerados
  hecho puente «God's Gift to Women was directed by Michael Curtiz»: σ = 0.554, rango 1, en Top-40: True, en K(q): True
  hecho puente «Edith Carlmar is known for the film 'Aldri»: σ = 0.480, rango 5, en Top-40: True, en K(q): True


In [12]:
sims, apr = etapa_filtro(Q, qv, est)

  S(q): 3 hechos aprobados; para ellos σ̃ = 1 (μ = 1 en el paseo):
    · God's Gift to Women was directed by Michael Curtiz.   ents = ["can:god's gift to women", 'can:michael curtiz']
    · Edith Carlmar is known for the film 'Aldri annet enn bråk'.   ents = ['can:edith carlmar', 'can:aldri annet enn bråk']
    · Edith Carlmar was Norway's first female film director.   ents = ['can:edith carlmar', 'can:norway']


### 5. Siembra: el vector $\mathbf v$ fuente a fuente (§2.4.5)

La presa densa lleva casi toda la masa, pero repartida en 6 118 pasajes; las fuentes estructurales concentran la suya en una decena de nodos.

In [13]:
pers = etapa_siembra(Q, qv, sims, apr, oro)

  masa sin normalizar ‖v‖₁ = 126.97: (a) hechos 5.00 · (b) presa densa 119.82 (94 % de la masa, en 6118 pasajes) · (c) anclas 0.10 · (d) enlace + pasaje propio 2.05
                      nodo  (a)   (b)  (c)   (d)  total  v norm.    
   can:god's gift to women  1.0 0.000 0.05 0.000  1.050   0.0083    
        can:michael curtiz  1.0 0.000 0.05 0.000  1.050   0.0083    
         can:edith carlmar  1.0 0.000 0.00 0.000  1.000   0.0079    
  can:aldri annet enn bråk  1.0 0.000 0.00 0.000  1.000   0.0079    
                can:norway  1.0 0.000 0.00 0.000  1.000   0.0079    
 chunk:God's Gift to Women  0.0 0.050 0.00 0.525  0.575   0.0045 ORO
      chunk:Michael Curtiz  0.0 0.030 0.00 0.525  0.555   0.0044 ORO
       chunk:Edith Carlmar  0.0 0.046 0.00 0.500  0.546   0.0043 ORO
chunk:Aldri annet enn bråk  0.0 0.045 0.00 0.500  0.545   0.0043 ORO
        chunk:Winter Light  0.0 0.046 0.00 0.000  0.046   0.0004    
  comprobación contra _semillas() del sistema: diferencia máxima 0.0e+00


### 6. Paseo con transiciones moduladas (§2.4.6)

Primero, dos filas reales de la matriz de transición $P$ (fija frente a modulada por $\mu_a$); después el orden final y, como contrafáctico, el mismo paseo con $\mu\equiv1$.

In [14]:
filas_transicion("can:god's gift to women", sims); print(); filas_transicion("can:michael curtiz", sims)

  can:god's gift to women: 10 salidas · transición FIJA (sin modular) = 0.100 para todas · MODULADA:
    μ = 1.000   P = 0.150   PASAJE  God's Gift to Women
    μ = 1.000   P = 0.150   HECHO   God's Gift to Women was directed by Michael Curtiz.
    μ = 0.637   P = 0.096   HECHO   God's Gift to Women was originally completed as a musical fi
    μ = 0.637   P = 0.096   HECHO   God's Gift to Women was released in 1931.
    μ = 0.630   P = 0.095   HECHO   God's Gift to Women stars Joan Blondell.
    μ = 0.608   P = 0.091   HECHO   God's Gift to Women is based on the play The Devil Was Sick.

  can:michael curtiz: 48 salidas · transición FIJA (sin modular) = 0.021 para todas · MODULADA:
    μ = 1.000   P = 0.039   PASAJE  God's Gift to Women
    μ = 1.000   P = 0.039   HECHO   God's Gift to Women was directed by Michael Curtiz.
    μ = 1.000   P = 0.039   PASAJE  Michael Curtiz
    μ = 1.000   P = 0.039   PASAJE  Prisoner of the Night (film)
    μ = 1.000   P = 0.039   PASAJE  The Vagabond 

In [15]:
top = etapa_paseo(pers, sims, oro); metricas(top, oro)

 puesto                        pasaje     p*    
      1                 Edith Carlmar 0.0070 ORO
      2           God's Gift to Women 0.0070 ORO
      3          Aldri annet enn bråk 0.0057 ORO
      4                Michael Curtiz 0.0054 ORO
      5                 Altid ballade 0.0013    
      6                   Anno Museum 0.0007    
      7       Frederick IV of Denmark 0.0007    
      8          Bedre enn sitt rykte 0.0007    
      9               Hans Bollandsås 0.0006    
     10 Louise of Mecklenburg-Güstrow 0.0006    
  R@2 = 0.50, FC@2 = 0   R@5 = 1.00, FC@5 = 1   R@10 = 1.00, FC@10 = 1


In [16]:
print("contrafáctico: misma siembra, matriz de transición FIJA (μ ≡ 1):")
top_fijo = etapa_paseo(pers, sims, oro, fijo=True); metricas(top_fijo, oro)

contrafáctico: misma siembra, matriz de transición FIJA (μ ≡ 1):
 puesto                        pasaje     p*    
      1           God's Gift to Women 0.0067 ORO
      2                 Edith Carlmar 0.0067 ORO
      3          Aldri annet enn bråk 0.0057 ORO
      4                Michael Curtiz 0.0056 ORO
      5                 Altid ballade 0.0011    
      6       Frederick IV of Denmark 0.0007    
      7                   Anno Museum 0.0007    
      8          Bedre enn sitt rykte 0.0006    
      9               Hans Bollandsås 0.0005    
     10 Louise of Mecklenburg-Güstrow 0.0005    
  R@2 = 0.50, FC@2 = 0   R@5 = 1.00, FC@5 = 1   R@10 = 1.00, FC@10 = 1


## Pregunta de comparación (estrategia de descomposición, $J>0$)

*«Are Christopher Newton (Criminal) and Frances M. Vega of the same nationality?»* — el enrutador elige la descomposición:
una llamada genera subconsultas, cada una aporta un vector, y $\sigma_a$ es el máximo sobre ellos. No corre el filtro:
la siembra usa la compuerta blanda.

In [17]:
q2 = buscar_pregunta("Are Christopher Newton"); Q2 = q2["question"]; oro2 = titulos_oro(q2)
print(Q2); print("pasajes oro:", sorted(oro2))
est2 = etapa_enrutador(Q2); L2 = etapa_enlazador(Q2)

Are Christopher Newton (Criminal) and Frances M. Vega of the same nationality?
pasajes oro: ['Christopher Newton (criminal)', 'Frances M. Vega']
tipo léxico: comparison  →  estrategia: descomposicion  (descomposición en subconsultas, 1 llamada; sin filtro)
  tramo 'Christopher Newton (Criminal) and Frances M Vega' → plegado 'christopher newton and frances m vega' → alias exacto: can:christopher newton
  L(q) = ['can:christopher newton', 'can:frances m. vega']


In [18]:
qv2, sig2 = etapa_afinidad(Q2, est2, n=10)
print("\nsubconsultas generadas (v_1 … v_J):"); 
# _qvecs no devuelve el texto de las subconsultas; se muestra el efecto: qué vector gana cada hecho (columna 'vector')

  J = 3 vectores adicionales · σ_a = max_j <v_j, e(τ_a)> sobre 47999 hechos: mín -0.074, máx 0.730
    σ vector                                                                   hecho τ_a                                     entidades
0.730    v_2                           Frances Marie Vega was born on September 2, 1983.                         [can:frances m. vega]
0.723    v_2                             Frances Marie Vega was of Puerto Rican descent.                         [can:frances m. vega]
0.692    v_1                        Christopher J. Newton was born on November 13, 1969.                      [can:christopher newton]
0.667    v_1                             Christopher J. Newton was an American murderer.                      [can:christopher newton]
0.664    v_2                        Frances Marie Vega was a United States Army soldier. [can:frances m. vega, can:united states army]
0.656    v_1                                             Christopher Newton is Canadian.   

In [19]:
sims2, apr2 = etapa_filtro(Q2, qv2, est2)
pers2 = etapa_siembra(Q2, qv2, sims2, apr2, oro2, top=11)

  la estrategia de descomposición no ejecuta el filtro: S(q) = ∅ y σ̃ = σ (compuerta blanda en la siembra)
  masa sin normalizar ‖v‖₁ = 142.49: (a) hechos 2.71 · (b) presa densa 138.06 (97 % de la masa, en 6118 pasajes) · (c) anclas 0.40 · (d) enlace + pasaje propio 1.32
                               nodo   (a)   (b)  (c)   (d)  total  v norm.    
                can:frances m. vega 0.954 0.000 0.15 0.000  1.104   0.0077    
                   can:david newton 0.881 0.000 0.05 0.000  0.931   0.0065    
              chunk:Frances M. Vega 0.000 0.050 0.00 0.552  0.602   0.0042 ORO
             can:christopher newton 0.451 0.000 0.15 0.000  0.601   0.0042    
        chunk:David Newton (artist) 0.000 0.044 0.00 0.465  0.509   0.0036    
           chunk:Christopher Newton 0.000 0.050 0.00 0.301  0.351   0.0025    
                        can:oakland 0.294 0.000 0.00 0.000  0.294   0.0021    
                 can:north carolina 0.127 0.000 0.00 0.000  0.127   0.0009    
                 

In [20]:
top2 = etapa_paseo(pers2, sims2, oro2); metricas(top2, oro2)
print("\nla entrada enlazada está FUNDIDA (dos personas): ")
print(json.dumps(entrada("can:christopher newton"), ensure_ascii=False, indent=1))

 puesto                        pasaje     p*    
      1               Frances M. Vega 0.0065 ORO
      2         David Newton (artist) 0.0057    
      3            Christopher Newton 0.0028    
      4 Christopher Newton (criminal) 0.0011 ORO
      5                        Sutekh 0.0004    
      6            Herman C. Raymaker 0.0004    
      7                  Victor Nuñez 0.0003    
      8           Christopher O'Neill 0.0003    
      9              Christine Pascal 0.0003    
     10                  Serena Evans 0.0003    
  R@2 = 0.50, FC@2 = 0   R@5 = 1.00, FC@5 = 1   R@10 = 1.00, FC@10 = 1

la entrada enlazada está FUNDIDA (dos personas): 
{
 "id": "can:christopher newton",
 "nombre": "Christopher Newton",
 "alias": [
  "Christopher J. Newton",
  "Christopher Newton",
  "Christopher Newton (criminal)"
 ],
 "pasajes": [
  "Christopher Newton",
  "Christopher Newton (criminal)"
 ],
 "pasaje_propio": "Christopher Newton",
 "n_menciones": 3,
 "descripcion": "Christopher Newton

## Pregunta composicional (la traza de extremo a extremo del documento, §2.5)

*«Who is the mother of the director of film Atomised (Film)?»* — todas las etapas seguidas.

In [21]:
q3 = buscar_pregunta("Who is the mother of the director of film Atomised"); Q3 = q3["question"]; oro3 = titulos_oro(q3)
print(Q3, "| oro:", sorted(oro3))
est3 = etapa_enrutador(Q3); L3 = etapa_enlazador(Q3)
qv3, sig3 = etapa_afinidad(Q3, est3, n=5)
K3 = etapa_candidatos(L3, sig3, puentes=("Atomised was directed by Oskar Roehler",))
sims3, apr3 = etapa_filtro(Q3, qv3, est3)
pers3 = etapa_siembra(Q3, qv3, sims3, apr3, oro3, top=6)
top3 = etapa_paseo(pers3, sims3, oro3, k=5); metricas(top3, oro3)

Who is the mother of the director of film Atomised (Film)? | oro: ['Atomised (film)', 'Oskar Roehler']
tipo léxico: compositional  →  estrategia: juez  (filtro selector, 1 llamada)
  tramo 'Atomised' → plegado 'atomised' → alias exacto: can:atomised
  L(q) = ['can:atomised']
  J = 0 vectores adicionales · σ_a = max_j <v_j, e(τ_a)> sobre 47999 hechos: mín -0.100, máx 0.503
    σ vector                                                                   hecho τ_a                                                                       entidades
0.503    v_0                                     Atomised was directed by Oskar Roehler.                                               [can:atomised, can:oskar roehler]
0.503    v_0                             Allison Anders is an independent film director.                                                            [can:allison anders]
0.486    v_0                    Allison Anders is an American independent film director.                              

## 7. Las mismas preguntas en HippoRAG 2 y en las dos versiones nuestras (resultados guardados, §5.3)

Los ficheros `artifacts/wiki2/resultados_*.jsonl` guardan el top-20 de cada sistema por pregunta (medición única sobre el banco).

In [22]:
W = ROOT / "artifacts" / "wiki2"
def top_guardado(nombre, qid): return next(json.loads(l) for l in open(W / f"resultados_{nombre}.jsonl", encoding="utf-8") if json.loads(l)["_id"] == qid)
for qq, oo in [(q, oro), (q2, oro2), (q3, oro3)]:
    print("=" * 100); print(qq["question"])
    for nombre, etiqueta in [("hipporag2_benchmark", "HippoRAG 2 (su código, mismos modelos)"), ("catrag_router_v3dicc_benchmark", "CatRAG-Reif v3"), ("plan_deepseek_benchmark", "Plan A (deepseek-chat)")]:
        f = top_guardado(nombre, qq["_id"])
        print(f"  {etiqueta:40} FC@5 = {f['full_chain@5']:.0f}  R@5 = {f['recall@5']:.2f}   top-5: " + " | ".join(f"{t}{' ★' if t in oo else ''}" for t in f["top"][:5]))

Which film has the director who is older, God'S Gift To Women or Aldri Annet Enn Bråk?
  HippoRAG 2 (su código, mismos modelos)   FC@5 = 0  R@5 = 0.75   top-5: Aldri annet enn bråk ★ | God's Gift to Women ★ | Edith Carlmar ★ | Altid ballade | Bedre enn sitt rykte
  CatRAG-Reif v3                           FC@5 = 1  R@5 = 1.00   top-5: Edith Carlmar ★ | God's Gift to Women ★ | Aldri annet enn bråk ★ | Michael Curtiz ★ | Altid ballade
  Plan A (deepseek-chat)                   FC@5 = 1  R@5 = 1.00   top-5: God's Gift to Women ★ | Edith Carlmar ★ | Aldri annet enn bråk ★ | Michael Curtiz ★ | Altid ballade
Are Christopher Newton (Criminal) and Frances M. Vega of the same nationality?
  HippoRAG 2 (su código, mismos modelos)   FC@5 = 1  R@5 = 1.00   top-5: Frances M. Vega ★ | Christopher Newton | Christopher Newton (criminal) ★ | David Newton (artist) | Chloé Robichaud
  CatRAG-Reif v3                           FC@5 = 1  R@5 = 1.00   top-5: Frances M. Vega ★ | David Newton (artist) | Christ

## Prueba con una pregunta propia

Cambia `Q_libre` (en inglés, sobre el corpus de 2Wiki). Coste: una incrustación y una llamada al filtro.

La pregunta de ejemplo enseña de paso el fallo medido del enlazador (§2.4.2): el tramo «Goose Woman» pierde el artículo y no casa con el alias «The Goose Woman», así que $L(q)=\varnothing$; la pregunta se resuelve igualmente porque el hecho puente entra por el Top-40 y el filtro lo aprueba.

In [23]:
Q_libre = "Who is the father of the director of film The Goose Woman?"
estL = etapa_enrutador(Q_libre); LL = etapa_enlazador(Q_libre)
qvL, sigL = etapa_afinidad(Q_libre, estL, n=5)
simsL, aprL = etapa_filtro(Q_libre, qvL, estL)
persL = etapa_siembra(Q_libre, qvL, simsL, aprL, set(), top=6)
_ = etapa_paseo(persL, simsL, set(), k=5)

tipo léxico: compositional  →  estrategia: juez  (filtro selector, 1 llamada)
  tramo 'Goose Woman' → plegado 'goose woman' → alias exacto: None
  L(q) = []
  J = 0 vectores adicionales · σ_a = max_j <v_j, e(τ_a)> sobre 47999 hechos: mín -0.081, máx 0.660
    σ vector                                                               hecho τ_a                                     entidades
0.660    v_0 The Goose Woman is a 1925 silent film drama directed by Clarence Brown.     [can:the goose woman, can:clarence brown]
0.658    v_0                     The Goose Woman was released by Universal Pictures. [can:the goose woman, can:universal pictures]
0.640    v_0                          The Goose Girl was directed by Fritz Genschow.      [can:the goose girl, can:fritz genschow]
0.636    v_0                         The Goose Woman stars Jack Pickford as her son.      [can:the goose woman, can:jack pickford]
0.628    v_0                                   The Goose Woman was released in 1925.     

## Evaluación por lotes con el harness (opcional)

`evaluar(buscar, preguntas)` calcula R@k y FC@k por pregunta y agregados con intervalos bootstrap. Desactivado por defecto:
ponlo a `True` para medir 20 preguntas del sondeo (≈ 0,02 USD, un par de minutos).

In [24]:
EJECUTAR_LOTE = False
if EJECUTAR_LOTE:
    from asistente_vih.eval.wiki2 import cargar_sondeo, evaluar, imprimir
    rs = Wiki2CatRAG(split="sondeo", variante="router", modo="paridad", canonico=True)
    preg_s, _ = cargar_sondeo()
    agg = evaluar(rs.buscar, preg_s[:20], nombre="cuaderno_sondeo_20", guardar=False)
    imprimir(agg)
else:
    print("lote desactivado (EJECUTAR_LOTE = False)")

lote desactivado (EJECUTAR_LOTE = False)


## Correspondencia con el documento

- Índice (entrada canónica, aserción, grafo por capas): §2.2, §2.3.2–§2.3.4.
- Enrutador: §2.4.1 · Enlazador: §2.4.2 (medición del escalón 2 y evoluciones L1–L4) · Afinidad y $J$: §2.4.3.
- $\mathcal K(q)$, frontera y filtro: §2.4.4 (medición de la frontera sobre el banco) · Siembra y desglose por fuentes: §2.4.5 (tablas 12 y 13).
- Paseo: §2.4.6 (ec. de modulación, forma cerrada en tres canales, ejemplo numérico §2.4.7) · Traza de extremo a extremo: §2.5 (figura 7).
- Comparación con HippoRAG 2 y CatRAG: §1.3 (las dos palancas) y §5.